 # Pruebas Unitarias con Python
 — [*Mario Abarca*](mailto:knkillname@gmail.com)

<a href="https://colab.research.google.com/github/knkillname/exploraciones/blob/master/cuadernos/Pruebas%20Unitarias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

 ## 1. Principios de las Pruebas Unitarias

 **Las pruebas manuales**…
 - son molestas
 - producen muchas falsas alarmas
 - se sienten innecesarias (por exceso de confianza)

 Este es un pseudocódigo de cómo suelen realizarse las pruebas manuales:

 ---
     PASO 1: Buscar errores en el código fuente del programa
     PASO 2: Un usuario manipula algo en el programa
     PASO 3: ¿Funcionó?
             El usuario dice que no: volver al PASO 1
 ---

 Las [**pruebas unitarias**][1] (_unit testing_) son un marco conceptual para
 las pruebas de software (y por lo tanto para el
 [desarrollo de software][2]).
 Cada **prueba unitaria**…
 - es una prueba automatizada
 - verifica un aspecto único y específico del código del programa (*unidad*)
 - responde una sola pregunta de sí/no sobre la funcionalidad de la unidad

 [1]: https://en.wikipedia.org/wiki/Unit_testing
 [2]: https://en.wikipedia.org/wiki/Software_development

 A continuación un ejemplo burdo e inocente de una prueba unitaria para una
 función que supuestamente incrementa cualquier entero no negativo en 1:

In [ ]:
# ADVERTENCIA: Solo para fines educativos. No intenten esto en casa.
def increment(y):
    """
    Devuelve y + 1.
    """
    if y == 0:
        return 1
    if y % 2 == 1:
        return 2 * increment(y // 2)
    return y + 1



In [ ]:
def test_increment():
    """
    Verifica si increment(y) realmente funciona como se espera.
    """
    if increment(0) != 1:
        return False  # La prueba falla cuando la entrada es 0
    if increment(7) != 8:
        return False  # La prueba falla con un número impar
    if increment(8) != 9:
        return False  # La prueba falla con un número par
    return True  # La prueba pasó


test_increment()


 Aunque las pruebas suelen realizarse como una idea tardía, las pruebas
 unitarias conducen naturalmente al
 [**Desarrollo Guiado por Pruebas**][1] (_Test-Driven Development_) que
 funciona así:

 ---
     PASO 1: Crear una prueba para la funcionalidad que se quiere implementar
             (fallará o incluso dará error en esta etapa)
     PASO 2: Escribir o depurar el código para que la prueba pase
     PASO 3: Ejecutar todas las pruebas
     PASO 4: ¿Pasan todas las pruebas? Si SÍ ir al PASO 1, si NO ir al PASO 2
 ---

 [1]: https://en.wikipedia.org/wiki/Test-driven_development

 Al menos en teoría, este enfoque de desarrollo guiado por pruebas resulta más
 gratificante para el programador, aliviando así la ansiedad asociada a las
 pruebas.

 Observa cómo la prueba es una función booleana (es decir, `True`/`False`).
 Una prueba nunca debe lanzar un error para señalar que falló.

 Como lo expresó [Kent Beck][1]:
 > ### Fallos y Errores
 > El framework distingue entre fallos y errores.
 > Un fallo es un problema anticipado.
 > Cuando escribes pruebas, verificas resultados esperados.
 > Si obtienes una respuesta diferente, eso es un fallo.
 > Un error es más catastrófico, una condición de error que no previste.

 [1]: https://web.archive.org/web/20150315073817/http://www.xprogramming.com/testfram.htm

 En nuestro siguiente ejemplo verificamos si `increment` funciona como se espera:
 - Esperamos que `increment` lance un `ValueError` para todos los valores
   negativos de `y`.
 - Si `increment` no lanza un `ValueError`, eso sería un fallo de prueba,
   no un error de prueba.

In [ ]:
def test_increment_negative():
    """
    Verifica si increment(y) lanza un error cuando y < 0
    """
    try:
        increment(-1)
    except ValueError:
        return True  # La prueba pasa si el error es un ValueError
    except:
        return False  # Cualquier otro error hace que la prueba falle
    return False  # No lanzar un error hace que la prueba falle


test_increment_negative()


 La prueba falla porque si intentas incrementar un número negativo se produce un
 `RecursionError`, no un `ValueError` como se muestra a continuación:

In [ ]:
increment(-1)


 Arreglemos el código para que esta prueba pueda pasar:

In [ ]:
def increment(y):
    """
    Devuelve y + 1.
    """
    if y < 0:
        raise ValueError()  # <- Este es el código para que la nueva prueba pase
    if y == 0:
        return 1
    if y % 2 == 1:
        return 2 * increment(y // 2)
    return y + 1



In [ ]:
# Esto debería lanzar un ValueError ahora:
increment(-1)


In [ ]:
# Ahora la prueba debería pasar:
test_increment_negative()


 Por supuesto, un programador debe anticipar tantos casos como sea posible para
 que las pruebas sean exitosas y exhaustivas.
 Por ejemplo: ¿qué pasaría si intentamos incrementar un número de punto
 flotante como $1.5$?

 ## 2. Pruebas con el Paquete `unittest`

 `unittest` es el marco de pruebas integrado de Python. Requiere que:
 - Coloques tus pruebas en clases como métodos
 - Uses una serie de métodos de aserción especiales en la clase `unittest.TestCase`
   en lugar de la [instrucción `assert`][1] incorporada

 [1]: https://realpython.com/lessons/assertions-and-tryexcept/

 ### 2.1 Un ejemplo básico
 Así es como se ve un script típico de pruebas unitarias en Python:

In [ ]:
# Código para tests/test_string.py
import unittest


class TestStringMethods(unittest.TestCase):  # TestCase es la clase base

    def test_upper(self):  # Los nombres de los métodos de prueba empiezan con "test_"
        self.assertEqual("foo".upper(), "FOO")  # Verifica un resultado esperado

    def test_isupper(self):
        self.assertTrue("FOO".isupper())  # Verifica una condición
        self.assertFalse("Foo".isupper())

    def test_split(self):
        s = "hello world"
        self.assertEqual(s.split(), ["hello", "world"])
        # Verifica que se lance una excepción específica:
        with self.assertRaises(TypeError):
            s.split(2)  # Esto debería lanzar TypeError porque 2 no es un str


if __name__ == "__main__":
    # Descubre y ejecuta automáticamente todas las pruebas
    unittest.main(argv=[""], exit=False)


 Para ejecutar todas las pruebas de un proyecto dado podemos usar el intérprete
 de línea de comandos (p. ej. [Anaconda prompt][1]):
 ```bash
 cd directorio_del_proyecto
 python -m unittest
 ```

 El primer comando cambia el directorio de trabajo actual al directorio del
 proyecto (debes reemplazar `directorio_del_proyecto` con el directorio real de
 tu proyecto) y el segundo inicia `unittest` de Python con
 [Descubrimiento de Pruebas][2] (_Test Discovery_) para que encuentre y ejecute
 automáticamente todas las pruebas del proyecto dado.

 [1]: https://docs.anaconda.com/anaconda/user-guide/getting-started/#open-anaconda-prompt
 [2]: https://docs.python.org/3/library/unittest.html#unittest-test-discovery

 **NOTA** Hemos llamado a `unittest.main` con los parámetros `argv=['']` y
 `exit=False` [porque estamos usando un Jupyter Notebook][1].
 Normalmente escribirías esto sin argumentos, así:
 ```python
 if __name__ == '__main__':
     unittest.main()
 ```

 [1]: https://medium.com/@vladbezden/using-python-unittest-in-ipython-or-jupyter-732448724e31

 Aquí hay una lista de los métodos de aserción que encontrarás en un `TestCase`:

 |Método                     |Verifica que          |
 |---------------------------|----------------------|
 |`assertEqual(a, b)`        |`a == b`              |
 |`assertNotEqual(a, b)`     |`a != b`              |
 |`assertTrue(x)`            |`bool(x) es True`     |
 |`assertFalse(x)`           |`bool(x) es False`    |
 |`assertIs(a, b)`           |`a is b`              |
 |`assertIsNot(a, b)`        |`a is not b`          |
 |`assertIsNone(x)`          |`x is None`           |
 |`assertIsNotNone(x)`       |`x is not None`       |
 |`assertIn(a, b)`           |`a in b`              |
 |`assertNotIn(a, b)`        |`a not in b`          |
 |`assertIsInstance(a, b)`   |`isinstance(a, b)`    |
 |`assertNotIsInstance(a, b)`|`not isinstance(a, b)`|

In [ ]:
# Eliminar los métodos de prueba de la memoria para no ejecutarlos de nuevo en
# el resto de ejemplos de este cuaderno.
# Por favor, abstente de hacer esto en código de producción:
del TestStringMethods


 ### 2.2 Ejemplo de Desarrollo Guiado por Pruebas usando `unittest`

 Para este ejercicio implementaremos [Kata09: Back to the Checkout][1]
 (las Katas son ejercicios de programación populares para practicar).
 La idea es desarrollar un *Simulador de Caja Registradora 2020*:
 > Una caja de supermercado que calcula el precio total de varios artículos.
 > En un supermercado normal, los productos se identifican mediante Unidades
 > de Mantenimiento de Stock, o SKUs (_Stock Keeping Units_). En nuestra
 > tienda, usaremos letras individuales del alfabeto (A, B, C, etc.).
 > Nuestros productos tienen precio individual.

 [1]: http://codekata.com/kata/kata09-back-to-the-checkout/

 Por lo tanto, la interfaz debería verse así:
 ```python
 checkout = Checkout(prices)
 checkout.scan(item)
 checkout.scan(item)
     ⋮
 checkout.total()
 ```

 Comenzamos en el PASO 1 definiendo un caso de prueba donde instanciamos una
 caja registradora con algunos precios de SKU al inicio de la prueba:

In [ ]:
import unittest


class TestCheckout(unittest.TestCase):

    def test_can_get_correct_total(self):
        checkout = Checkout({"A": 1.99, "B": 2.50, "C": 1.50})
        checkout.scan("A")
        checkout.scan("A")
        checkout.scan("B")
        checkout.scan("A")
        checkout.scan("C")
        total = checkout.total()
        self.assertEqual(total, 9.97)



 Si intentas ejecutar `unittest.main` obtendrás un error
 (no un fallo) porque ni siquiera hemos definido una clase `Checkout`.

In [ ]:
unittest.main(argv=[""], exit=False)


 Pasemos al PASO 2: escribamos el código mínimo para implementar `Checkout` y
 hacer que pase la prueba:

In [ ]:
class Checkout:
    def __init__(self, prices):
        self._prices = prices
        self._total = 0.0

    def scan(self, item):
        self._total += self._prices[item]

    def total(self):
        return round(self._total, 2)



 Bien, ahora en el PASO 3 ejecutamos las pruebas de nuevo:

In [ ]:
# Puedes ejecutar pruebas con información más detallada pasando el argumento
# verbosity así:
unittest.main(argv=[""], exit=False, verbosity=2)


 Ahora que todas las pruebas pasan, el PASO 4 nos redirige al PASO 1.
 ¿Hay alguna nueva funcionalidad que queramos implementar?
 Pues sí, de hecho:

 > algunos artículos tienen precio múltiple: compra $n$ de ellos y te costarán
 > $y$ centavos. Por ejemplo, el artículo *A* podría costar 50 centavos
 > individualmente, pero esta semana tenemos una oferta especial: compra tres
 > *A*'s y te costarán $1.30

In [ ]:
import unittest


class TestCheckout(unittest.TestCase):
    # unittest llamará automáticamente a setUp antes de cada prueba que ejecutemos.
    # De esta forma no hay necesidad de crear un checkout en cada prueba.
    def setUp(self):
        prices = {
            "A": Discount(1.99, 3, 6.69),  # $1.99 c/u o 3 por $6.69
            "B": Discount(2.50, 2, 5.80),  # $2.50 c/u o 2 por $5.80
            "C": 1.50,  # $1.50 c/u
        }
        self.checkout = Checkout(prices)

    def test_total_without_discount(self):
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("B")
        self.checkout.scan("C")
        total = self.checkout.total()
        self.assertEqual(total, 7.98)

    def test_total_with_discount(self):
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("B")
        self.checkout.scan("A")
        self.checkout.scan("A")
        self.checkout.scan("C")
        total = self.checkout.total()
        self.assertEqual(total, 21.36)



 Pasemos al PASO 2:

In [ ]:
from typing import NamedTuple


class Discount(NamedTuple):
    unit_price: float
    items_per_discount: int
    discounted_price: float


class Checkout:
    def __init__(self, prices):
        self._prices = prices
        self._count = {}

    def scan(self, item):
        if item not in self._count.keys():
            self._count[item] = 0
        self._count[item] += 1

    def total(self):
        total = 0.0
        for sku, count in self._count.items():
            price = self._prices[sku]
            if isinstance(price, Discount):
                discounts, remaining = divmod(count, price.items_per_discount)
                price = (
                    discounts * price.discounted_price + remaining * price.unit_price
                )
            total += price
        return total



 Ahora ejecutemos el PASO 3:

In [ ]:
unittest.main(argv=[""], exit=False, verbosity=2)


 ## 3. Usar Ejemplos de Documentación como Pruebas con `doctest`

 Si alguna vez has usado el intérprete de Python en la línea de comandos o
 [IDLE][1] ya has visto Python en el [Modo Interactivo][2].
 En este modo el intérprete imprime un mensaje de bienvenida indicando su
 número de versión y un aviso de derechos de autor antes de solicitar el
 siguiente comando usando tres signos de mayor que (`>>>`).
 Cada vez que ingresas un comando, este se ejecuta y la salida se muestra
 debajo. Se ve más o menos así:

 ```
 Python 3.7.5 (default, Nov 20 2019, 09:21:52)
 [GCC 9.2.1 20191008] on linux
 Type "help", "copyright", "credits" or "license()" for more information.
 ```
 ```python
 >>> cubes = [1, 8, 27, 65, 125]  # algo está mal aquí
 >>> 4 ** 3  # ¡el cubo de 4 es 64, no 65!
 64
 >>> cubes[3] = 64  # reemplazar el valor incorrecto
 >>> cubes
 [1, 8, 27, 64, 125]
 ```

 **NOTA**: Los Jupyter Notebooks expanden la idea del modo interactivo con una
 interfaz más elaborada.

 Entonces, ¿qué tiene que ver el modo interactivo con las pruebas unitarias?
 Veamos un ejemplo anterior en modo interactivo:

 [1]: https://en.wikipedia.org/wiki/IDLE
 [2]: https://docs.python.org/3/tutorial/interpreter.html#interactive-mode

In [ ]:
checkout = Checkout(
    {
        "A": Discount(1.99, 3, 6.69),  # $1.99 c/u o 3 por $6.69
        "B": Discount(2.50, 2, 5.80),  # $2.50 c/u o 2 por $5.80
        "C": 1.50,  # $1.50 c/u
    }
)
checkout.scan("A")
checkout.scan("A")
checkout.scan("A")
checkout.scan("A")
checkout.scan("A")
checkout.scan("A")
checkout.scan("B")
checkout.scan("A")
checkout.scan("A")
checkout.scan("C")
checkout.total()


 Como ves, el código anterior da un ejemplo de cómo usar la clase `Checkout` y
 también documenta cuál es la salida esperada (21.36), así que en cierto
 sentido esto equivale a una sola prueba «*si haces esto entonces deberías
 obtener este resultado*» sin siquiera usar el método `assertEqual`.
 Para convertir este ejemplo en una prueba real usamos el módulo `doctest`.

 Primero, necesitamos verificar cómo se vería este ejemplo *literalmente* en
 una shell interactiva real de Python.
 Puedes agregar los prompts manualmente, o puedes ejecutar Python en una línea
 de comandos real y pegarlo.
 De cualquier forma, deberías terminar con esto:

 ```python
 >>> checkout = Checkout({
 ...     'A': Discount(1.99, 3, 6.69),  # $1.99 c/u o 3 por $6.69
 ...     'B': Discount(2.50, 2, 5.80),  # $2.50 c/u o 2 por $5.80
 ...     'C': 1.50  # $1.50 c/u
 ... })
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('B')
 >>> checkout.scan('A')
 >>> checkout.scan('A')
 >>> checkout.scan('C')
 >>> checkout.total()
 21.36
 ```

 Ahora agreguemos este ejemplo a la documentación de `Checkout.total` usando un
 [docstring][1]:

 [1]: https://www.python.org/dev/peps/pep-0257/

In [ ]:
from typing import NamedTuple


class Discount(NamedTuple):
    unit_price: float
    items_per_discount: int
    discounted_price: float


class Checkout:
    def __init__(self, prices):
        self._prices = prices
        self._count = {}

    def scan(self, item):
        if item not in self._count.keys():
            self._count[item] = 0
        self._count[item] += 1

    def total(self):
        """
        Devuelve el precio total de los artículos escaneados después del descuento.

        Ejemplo de uso:
        >>> checkout = Checkout({
        ...     'A': Discount(1.99, 3, 6.69),  # $1.99 c/u o 3 por $6.69
        ...     'B': Discount(2.50, 2, 5.80),  # $2.50 c/u o 2 por $5.80
        ...     'C': 1.50  # $1.50 c/u
        ... })
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('B')
        >>> checkout.scan('A')
        >>> checkout.scan('A')
        >>> checkout.scan('C')
        >>> checkout.total()
        21.36
        """
        total = 0.0
        for sku, count in self._count.items():
            price = self._prices[sku]
            if isinstance(price, Discount):
                discounts, remaining = divmod(count, price.items_per_discount)
                price = (
                    discounts * price.discounted_price + remaining * price.unit_price
                )
            total += price
        return total



 Finalmente, ejecutamos la prueba invocando `doctest.testmod`:

In [ ]:
import doctest

doctest.testmod()


 Idealmente agregarías las siguientes líneas al final de cada módulo de Python
 que escribas:

 ```python
 if __name__ == '__main__':
     import doctest
     doctest.testmod()
 ```

 Mira el siguiente ejemplo:

In [ ]:
"""
Este es el módulo "example".

El módulo example proporciona una función, factorial(). Por ejemplo,

>>> factorial(5)
120
"""


def factorial(n):
    """Devuelve el factorial de n, un entero exacto >= 0.

    >>> [factorial(n) for n in range(6)]
    [1, 1, 2, 6, 24, 120]
    >>> factorial(30)
    265252859812191058636308480000000
    >>> factorial(-1)
    Traceback (most recent call last):
        ...
    ValueError: n debe ser >= 0

    Los factoriales de flotantes están bien, pero el flotante debe ser un entero exacto:
    >>> factorial(30.1)
    Traceback (most recent call last):
        ...
    ValueError: n debe ser entero exacto
    >>> factorial(30.0)
    265252859812191058636308480000000

    Tampoco debe ser ridículamente grande:
    >>> factorial(1e100)
    Traceback (most recent call last):
        ...
    OverflowError: n demasiado grande
    """

    import math

    if not n >= 0:
        raise ValueError("n debe ser >= 0")
    if math.floor(n) != n:
        raise ValueError("n debe ser entero exacto")
    if n + 1 == n:  # capturar un valor como 1e300
        raise OverflowError("n demasiado grande")
    result = 1
    factor = 2
    while factor <= n:
        result *= factor
        factor += 1
    return result


if __name__ == "__main__":
    import doctest

    doctest.testmod()


In [ ]:
doctest.testmod()
